# Week 5: Capstone Modeling Lane

**Lane:** CTR / Engagement Opportunity Scoring

I will train a compact Logistic Regression scorer to estimate whether a client-content page's CTR improves in April, then compare it with the Week 4 rule on the same client-grouped test split.

## 1) Method choice and why

I choose Logistic Regression because the label is binary, the feature set is small and interpretable, and the model produces a ranking probability without rewarding unnecessary complexity. The model is decision support for review prioritization, not a causal claim about metadata changes.

In [1]:
from pathlib import Path
import duckdb
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

workspace_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'data' / 'internship-warehouse').exists()), Path.cwd())
root = workspace_root / 'data' / 'internship-warehouse'
con = duckdb.connect()
march = (root / 'fact_content_daily_performance' / 'month=2026-03' / '*.parquet').as_posix()
april = (root / 'fact_content_daily_performance' / 'month=2026-04' / '*.parquet').as_posix()
features = con.sql(f"""SELECT client_hash_id, content_hash_id, SUM(gsc_impressions)::DOUBLE impressions, SUM(gsc_clicks)::DOUBLE / NULLIF(SUM(gsc_impressions),0) ctr, SUM(gsc_sum_position)::DOUBLE / NULLIF(SUM(gsc_impressions),0) avg_position, SUM(sessions_organic)::DOUBLE sessions_organic FROM read_parquet('{march}') WHERE gsc_data_available IS TRUE GROUP BY ALL""").df()
features['expected_ctr'] = 0.10 / features['avg_position'].clip(lower=1).pow(0.35)
features['ctr_gap'] = (features['expected_ctr'] - features['ctr']).clip(lower=0)
outcomes = con.sql(f"""SELECT client_hash_id, content_hash_id, SUM(gsc_clicks)::DOUBLE / NULLIF(SUM(gsc_impressions),0) future_ctr FROM read_parquet('{april}') WHERE gsc_data_available IS TRUE GROUP BY ALL""").df()
data = features.merge(outcomes, on=['client_hash_id','content_hash_id'], how='inner').dropna(subset=['future_ctr','ctr'])
data['label'] = (data['future_ctr'] > data['ctr']).astype(int)
data['group_bucket'] = pd.util.hash_pandas_object(data['client_hash_id'], index=False) % 5
train = data[data['group_bucket'] != 0].copy()
test = data[data['group_bucket'] == 0].copy()
feature_cols = ['impressions','ctr','avg_position','sessions_organic','ctr_gap']
model = make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), LogisticRegression(max_iter=200, class_weight='balanced'))
model.fit(train[feature_cols], train['label'])
test['model_score'] = model.predict_proba(test[feature_cols])[:, 1]
test['baseline_score'] = test['ctr_gap'] * np.log1p(test['impressions'])
def precision_at_10(frame, score_col):
    return frame.sort_values(score_col, ascending=False).head(10)['label'].mean()
comparison = pd.DataFrame({
    'method': ['Week 4 baseline', 'Logistic Regression'],
    'precision_at_10': [precision_at_10(test, 'baseline_score'), precision_at_10(test, 'model_score')],
})
print(f'Train rows: {len(train):,}; test rows: {len(test):,}; positive rate: {data.label.mean():.3f}')
comparison

Train rows: 149,740; test rows: 8,809; positive rate: 0.195


,method,precision_at_10
0,Week 4 baseline,0.4
1,Logistic Regression,0.3


## 2) Split design

The split is grouped by pseudonymized client: all rows from a client stay in either train or test. This reduces leakage from repeated client behavior and evaluates whether the ranking transfers to unseen clients. April supplies the outcome label; no April field enters the feature columns.

## 3) Train + compare vs baseline

The comparison table is computed on the same held-out clients and the same `Precision@10` metric. The baseline is the Week 4 CTR-gap rule; the model is only preferable if its measured queue precision improves without violating the time boundary.

In [2]:
test['error_type'] = np.select([test['model_score'] >= 0.5, test['label'].eq(1)], ['predicted_positive','missed_positive'], default='correct_negative')
print('Top model rows:')
print(test.sort_values('model_score', ascending=False)[['client_hash_id','content_hash_id','model_score','label','error_type']].head(5).to_string(index=False))
print('Model feature coefficients:')
coefs = pd.Series(model[-1].coef_[0], index=feature_cols).sort_values(ascending=False)
print(coefs.to_string())

Top model rows:
         client_hash_id          content_hash_id  model_score  label         error_type
client_65de48885f4ef01b content_62673eea26c31c17     0.985525      1 predicted_positive
client_3197e6291363b4db content_22588e765b93dfac     0.947927      0 predicted_positive
client_3197e6291363b4db content_0cfa3d9d68583bf1     0.939551      0 predicted_positive
client_3197e6291363b4db content_d6a559d9a93a3eac     0.895327      0 predicted_positive
client_3f0ce4d44fe94f3d content_0b628d4de0ca17cb     0.894307      0 predicted_positive
Model feature coefficients:
impressions         0.432916
sessions_organic   -0.039839
ctr_gap            -0.121382
ctr                -0.345841
avg_position       -0.436302


## 4) Errors and interpretation

Errors are expected because CTR is affected by SERP features, query intent, position, seasonality, and measurement availability. A false positive is a page the queue prioritizes but that does not improve in April; a false negative is an improving page the queue misses. Coefficients are directional associations, not causal effects.

## 5) Self-check

- Method choice is explained.
- Validation is grouped by client.
- Baseline and model use the same held-out clients and `Precision@10`.
- Feature interpretation and error examples are printed.
- No future-window field is used as a feature.